# Handling missing values in the dataset using multiple imputation techniques, such as KNN imputation, MICE, and predictive modeling

In [3]:
import pandas as pd
import numpy as np


In [4]:
df = pd.read_csv('/home/patel-krish/anaconda3/BrainyBeam/womens_clothing/dataset/womens_clothing.csv')

print("Shape:", df.shape)
print("\nExplicit NaN counts:")
print(df.isnull().sum())
df.head()

Shape: (1500, 10)

Explicit NaN counts:
Clothing ID                0
Age                        0
Title                      0
Review Text                0
Rating                     0
Recommended IND            0
Positive Feedback Count    0
Division Name              0
Department Name            0
Class Name                 0
dtype: int64


,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,868,39,Disappointed with fit,"Slightly looser than expected around the bust,...",5,1,0,General,Tops,Skirts
1,829,37,No Title Provided,Material feels premium and holds up very well ...,3,0,1,General,Intimate,Lounge
2,831,81,Flattering cut,"Slightly looser than expected around the bust,...",5,1,0,Initmates,Intimate,Blouses
3,872,34,No Title Provided,Material feels premium and holds up very well ...,5,1,1,Initmates,Jackets,Sweaters
4,868,29,No Title Provided,No Review Text Provided,5,1,12,General Petite,Intimate,Skirts


In [ ]:
for col in df.select_dtypes(include='object').columns:
    print(f"--- {col} ---")
    print(df[col].value_counts().head(5), "\n")

df['Title'] = df['Title'].replace('No Title Provided', np.nan)
df['Review Text'] = df['Review Text'].replace('No Review Text Provided', np.nan)

missing_summary = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})
print(missing_summary[missing_summary['Missing Count'] > 0])

--- Title ---
Title
No Title Provided          230
Flattering cut             175
Runs a bit small           173
Comfortable and stylish    163
Perfect staple piece       160
Name: count, dtype: int64 

--- Review Text ---
Review Text
Slightly looser than expected around the bust, but very comfortable overall.            383
Material feels premium and holds up very well after multiple washes.                    368
The color is true to the online photo, but the length was a bit short for my height.    356
Absolutely love the fit and texture of this piece. Highly recommend!                    352
No Review Text Provided                                                                  41
Name: count, dtype: int64 

--- Division Name ---
Division Name
Initmates         505
General Petite    498
General           497
Name: count, dtype: int64 

--- Department Name ---
Department Name
Dresses     277
Bottoms     259
Trend       248
Tops        245
Intimate    243
Name: count, dtype: int64 


In [6]:
from scipy.stats import f_oneway, chi2_contingency

num_cols = ['Age', 'Rating', 'Recommended IND', 'Positive Feedback Count']
cat_predictors = ['Rating', 'Recommended IND', 'Division Name', 'Department Name', 'Class Name', 'Clothing ID']

for target in ['Title', 'Review Text']:
    print(f"\n=== Missingness Mechanism Check: {target} ===")
    for col in num_cols:
        present = df.loc[df[target].notna(), col]
        missing = df.loc[df[target].isna(), col]
        _, p_val = f_oneway(present, missing)
        print(f"  {col}: present_mean={present.mean():.3f}, missing_mean={missing.mean():.3f}, p={p_val:.4f}")

    print(f"\n=== Association with Observed Values: {target} ===")
    for col in cat_predictors:
        ct = pd.crosstab(df[target].dropna(), df.loc[df[target].notna(), col])
        chi2, p_val, _, _ = chi2_contingency(ct)
        print(f"  {target} vs {col}: chi2={chi2:.2f}, p={p_val:.4f}")


=== Missingness Mechanism Check: Title ===
  Age: present_mean=50.790, missing_mean=47.483, p=0.0156
  Rating: present_mean=4.180, missing_mean=4.304, p=0.1204
  Recommended IND: present_mean=0.774, missing_mean=0.843, p=0.0184
  Positive Feedback Count: present_mean=2.357, missing_mean=2.426, p=0.7266

=== Association with Observed Values: Title ===
  Title vs Rating: chi2=31.58, p=0.2918
  Title vs Recommended IND: chi2=9.50, p=0.2189
  Title vs Division Name: chi2=19.05, p=0.1631
  Title vs Department Name: chi2=50.18, p=0.0464
  Title vs Class Name: chi2=55.55, p=0.7363
  Title vs Clothing ID: chi2=61.29, p=0.5377

=== Missingness Mechanism Check: Review Text ===
  Age: present_mean=50.300, missing_mean=49.659, p=0.8320
  Rating: present_mean=4.197, missing_mean=4.268, p=0.6880
  Recommended IND: present_mean=0.783, missing_mean=0.854, p=0.2761
  Positive Feedback Count: present_mean=2.357, missing_mean=2.732, p=0.3930

=== Association with Observed Values: Review Text ===
  Revie

In [7]:
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

cat_cols = ['Title', 'Review Text', 'Division Name', 'Department Name', 'Class Name']
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan)

df_enc = df.copy()
df_enc[cat_cols] = encoder.fit_transform(df[cat_cols])

def create_masked_split(data, target_col, frac=0.2, seed=42):
    rng = np.random.default_rng(seed)
    known_idx = data[data[target_col].notna()].index.to_numpy()
    mask_idx = rng.choice(known_idx, size=int(frac * len(known_idx)), replace=False)
    df_masked = data.copy()
    df_masked.loc[mask_idx, target_col] = np.nan
    y_true = data.loc[mask_idx, target_col].astype(int).values
    return df_masked, mask_idx, y_true

splits = {
    col: create_masked_split(df_enc, col)
    for col in ['Title', 'Review Text']
}

In [8]:
from sklearn.impute import KNNImputer

knn_preds = {}

for col, (df_masked, mask_idx, _) in splits.items():
    n_classes = int(df_enc[col].max()) + 1
    scaler = StandardScaler()
    scaled = scaler.fit_transform(df_masked)

    knn = KNNImputer(n_neighbors=5, weights='distance')
    imputed = scaler.inverse_transform(knn.fit_transform(scaled))
    imputed_df = pd.DataFrame(imputed, columns=df_masked.columns, index=df_masked.index)

    knn_preds[col] = np.clip(np.rint(imputed_df.loc[mask_idx, col]), 0, n_classes - 1).astype(int).values

In [10]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

mice_preds = {}

for col, (df_masked, mask_idx, _) in splits.items():
    n_classes = int(df_enc[col].max()) + 1
    scaler = StandardScaler()
    scaled = scaler.fit_transform(df_masked)

    mice = IterativeImputer(estimator=BayesianRidge(), max_iter=15, random_state=42)
    imputed = scaler.inverse_transform(mice.fit_transform(scaled))
    imputed_df = pd.DataFrame(imputed, columns=df_masked.columns, index=df_masked.index)

    mice_preds[col] = np.clip(np.rint(imputed_df.loc[mask_idx, col]), 0, n_classes - 1).astype(int).values

In [11]:
from sklearn.ensemble import RandomForestClassifier

rf_preds = {}

for col, (df_masked, mask_idx, _) in splits.items():
    feature_cols = [c for c in df_masked.columns if c != col]
    X_all = df_masked[feature_cols].copy()

    # Fill auxiliary missing values with mode during feature matrix preparation
    for c in feature_cols:
        if X_all[c].isna().any():
            X_all[c] = X_all[c].fillna(X_all[c].mode()[0])

    train_idx = df_masked[df_masked[col].notna()].index
    rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
    rf.fit(X_all.loc[train_idx], df_masked.loc[train_idx, col].astype(int))

    rf_preds[col] = rf.predict(X_all.loc[mask_idx])

In [12]:
from sklearn.metrics import accuracy_score, f1_score
from scipy.spatial.distance import jensenshannon

for col, (_, _, y_true) in splits.items():
    n_classes = int(df_enc[col].max()) + 1
    true_dist = np.bincount(y_true, minlength=n_classes) / len(y_true)

    rows = []
    for name, preds in [
        ('KNN Imputation', knn_preds[col]),
        ('MICE (IterativeImputer)', mice_preds[col]),
        ('Predictive Modeling (RF)', rf_preds[col])
    ]:
        pred_dist = np.bincount(preds, minlength=n_classes) / len(preds)
        rows.append({
            'Method': name,
            'Accuracy': round(accuracy_score(y_true, preds), 4),
            'Macro-F1': round(f1_score(y_true, preds, average='macro'), 4),
            'JS Divergence (lower=better)': round(float(jensenshannon(true_dist, pred_dist)), 4)
        })

    print(f"\n=== Benchmark Results: {col} ===")
    print(pd.DataFrame(rows).to_string(index=False))


=== Benchmark Results: Title ===
                  Method  Accuracy  Macro-F1  JS Divergence (lower=better)
          KNN Imputation    0.1220    0.0927                        0.4614
 MICE (IterativeImputer)    0.0945    0.0216                        0.7296
Predictive Modeling (RF)    0.1260    0.1258                        0.1296

=== Benchmark Results: Review Text ===
                  Method  Accuracy  Macro-F1  JS Divergence (lower=better)
          KNN Imputation    0.2784    0.1936                        0.3577
 MICE (IterativeImputer)    0.2715    0.1068                        0.6029
Predictive Modeling (RF)    0.2440    0.2423                        0.0542


In [ ]:
df_imputed = df.copy()

selected_methods = {
    'Title': 'Predictive Modeling (RF)',
    'Review Text': 'Predictive Modeling (RF)'
}

for target_col, method in selected_methods.items():
    feature_cols = [c for c in df_enc.columns if c != target_col]
    X_all = df_enc[feature_cols].copy()

    for c in feature_cols:
        if X_all[c].isna().any():
            X_all[c] = X_all[c].fillna(X_all[c].mode()[0])

    train_mask = df_enc[target_col].notna()
    miss_mask = df_enc[target_col].isna()

    rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
    rf.fit(X_all[train_mask], df_enc.loc[train_mask, target_col].astype(int))
    preds = rf.predict(X_all[miss_mask])

    categories = encoder.categories_[cat_cols.index(target_col)]
    df_imputed.loc[miss_mask, target_col] = [categories[p] for p in preds]

print("Remaining missing values after imputation:")
print(df_imputed.isnull().sum())

Remaining missing values after imputation:
Clothing ID                0
Age                        0
Title                      0
Review Text                0
Rating                     0
Recommended IND            0
Positive Feedback Count    0
Division Name              0
Department Name            0
Class Name                 0
dtype: int64


In [14]:
for col in ['Title', 'Review Text']:
    comparison = pd.DataFrame({
        'Before (%)': (df[col].value_counts(normalize=True) * 100).round(2),
        'After (%)': (df_imputed[col].value_counts(normalize=True) * 100).round(2)
    })
    comparison['Shift (pp)'] = (comparison['After (%)'] - comparison['Before (%)']).round(2)
    print(f"\n--- Distribution Check: {col} ---")
    print(comparison)

df_imputed.to_csv('womens_clothing_imputed.csv', index=False)
print("\nSaved cleaned dataset to 'womens_clothing_imputed.csv'.")


--- Distribution Check: Title ---
                         Before (%)  After (%)  Shift (pp)
Title                                                     
Beautiful fabric              10.55      10.00       -0.55
Comfortable and stylish       12.83      12.93        0.10
Disappointed with fit         12.20      12.33        0.13
Flattering cut                13.78      13.80        0.02
Great quality                 12.05      12.07        0.02
Love this dress!              12.36      12.20       -0.16
Perfect staple piece          12.60      12.80        0.20
Runs a bit small              13.62      13.87        0.25

--- Distribution Check: Review Text ---
                                                    Before (%)  After (%)  \
Review Text                                                                 
Slightly looser than expected around the bust, ...       26.25      26.13   
Material feels premium and holds up very well a...       25.22      25.33   
The color is true to the o